# LeetCode 1229 — Meeting Scheduler bằng Python

Link đề: https://leetcode.com/problems/meeting-scheduler/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Sorting

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề và chữ ký hàm được đối chiếu với bản mô tả công khai tại https://leetcode.doocs.org/en/lc/1229/ (method `minAvailableDuration(slots1, slots2, duration)`).

Skeleton LeetCode:

```python
class Solution:
    def minAvailableDuration(self, slots1: List[List[int]], slots2: List[List[int]], duration: int) -> List[int]:
        ...
```

Ghi chú: trả về khung giờ **sớm nhất** `[start, start + duration]` mà cả hai người cùng rảnh, hoặc `[]` nếu không có. Đầu vào **chưa chắc đã sắp xếp**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biết tính **giao của hai khoảng** (interval intersection): `[max(s1, s2), min(e1, e2)]`, khác rỗng khi `max ≤ min`.
2. Đi từ **thử mọi cặp khoảng** `O(n·m)` → **sắp xếp + hai con trỏ** `O((n+m) log(n+m))` → **gộp hai danh sách + so sánh kề nhau** / **heap**.
3. Hiểu quy tắc di chuyển con trỏ: **bỏ khoảng kết thúc sớm hơn** vì nó không thể giao với bất kỳ khoảng nào sau nữa của người kia.
4. Tận dụng giả thiết "các khoảng của **cùng một người** không giao nhau" để chứng minh cách gộp chỉ cần so **hai khoảng kề nhau**.
5. Tránh các lỗi: quên sắp xếp, di chuyển nhầm con trỏ, dùng `>` thay vì `>=`, trả `[start, end]` thay vì `[start, start + duration]`, quên lọc khoảng ngắn trong cách gộp.

## 2. Đề bài & ví dụ

Cho hai mảng khung giờ rảnh `slots1`, `slots2` của hai người và độ dài cuộc họp `duration`. Trả về **khung giờ sớm nhất** hợp với cả hai người và có độ dài đúng `duration`.

Nếu không có khung chung nào thoả, trả về mảng rỗng.

Một khung giờ là mảng hai phần tử `[start, end]` biểu diễn khoảng **đóng** từ `start` đến `end`. Đảm bảo **không có hai khung của cùng một người giao nhau**: với hai khung `[start1, end1]`, `[start2, end2]` của cùng một người, hoặc `start1 > end2` hoặc `start2 > end1`.

**Ví dụ 1:** `slots1 = [[10,50],[60,120],[140,210]], slots2 = [[0,15],[60,70]], duration = 8` → `[60,68]`.

**Ví dụ 2:** cùng `slots1`, `slots2`, `duration = 12` → `[]`.

**Ràng buộc:**
- `1 <= slots1.length, slots2.length <= 10^4`
- `slots1[i].length, slots2[i].length == 2`
- `slots1[i][0] < slots1[i][1]`, `slots2[i][0] < slots2[i][1]`
- `0 <= slots1[i][j], slots2[i][j] <= 10^9`
- `1 <= duration <= 10^6`

In [1]:
import heapq
import random
import time
from typing import List

S1 = [[10, 50], [60, 120], [140, 210]]
S2 = [[0, 15], [60, 70]]
examples = [
    (S1, S2, 8, [60, 68]),
    (S1, S2, 12, []),
]
for s1, s2, d, want in examples:
    print(f"duration={d} -> {want}")

duration=8 -> [60, 68]
duration=12 -> []


## 3. Kiến thức nền

### 3.1 Giao của hai khoảng

Hai khoảng đóng `[a1, b1]` và `[a2, b2]` giao nhau thành `[max(a1, a2), min(b1, b2)]` nếu `max(a1, a2) <= min(b1, b2)`; ngược lại rỗng. Độ dài phần giao (để họp) là `min(b1, b2) − max(a1, a2)`.

In [2]:
def intersect(x, y):
    lo, hi = max(x[0], y[0]), min(x[1], y[1])
    return [lo, hi] if lo <= hi else None


print(intersect([10, 50], [0, 15]))    # [10, 15] dài 5
print(intersect([60, 120], [60, 70]))  # [60, 70] dài 10
print(intersect([140, 210], [60, 70])) # None

[10, 15]
[60, 70]
None


### 3.2 "Khung họp dài `duration`" nghĩa là gì?

Cuộc họp chiếm `[t, t + duration]`. Nó nằm gọn trong phần giao `[lo, hi]` khi `lo <= t` và `t + duration <= hi`. Sớm nhất là `t = lo`, nên điều kiện tồn tại là `hi − lo >= duration` (dấu **bằng** vẫn được — cuộc họp vừa khít).

In [3]:
lo, hi = 60, 70
for d in [8, 10, 12]:
    print(d, "->", [lo, lo + d] if hi - lo >= d else [])

8 -> [60, 68]
10 -> [60, 70]
12 -> []


### 3.3 Sắp xếp list các cặp

`list.sort()` sắp các list `[start, end]` theo **thứ tự từ điển**: so `start` trước, bằng nhau mới so `end`. Đó đúng là thứ tự ta cần (theo thời gian bắt đầu).

In [4]:
a = [[60, 70], [0, 15], [10, 5]]
a.sort()
print(a)
print(sorted([[140, 210], [10, 50], [60, 120]], key=lambda x: x[0]))

[[0, 15], [10, 5], [60, 70]]
[[10, 50], [60, 120], [140, 210]]


### 3.4 `heapq` — hàng đợi ưu tiên (min-heap)

`heapq.heapify(list)` biến list thành heap `O(n)`; `heapq.heappop` lấy phần tử **nhỏ nhất** `O(log n)`. Phần tử là tuple/list thì so sánh theo từ điển.

In [5]:
h = [[60, 70], [0, 15], [10, 50]]
heapq.heapify(h)
print([heapq.heappop(h) for _ in range(3)])

[[0, 15], [10, 50], [60, 70]]


## 4. Ý tưởng nền tảng / trực giác

Vẽ trục thời gian ví dụ 1:

```
người 1:      [10────50]   [60──────120]   [140────210]
người 2: [0──15]           [60─70]
giao   :      [10─15]      [60─70]
```

- Giao `[10, 15]` dài 5 — không đủ 8.
- Giao `[60, 70]` dài 10 — đủ 8 → họp `[60, 68]` (bắt đầu sớm nhất có thể).
- Với `duration = 12`: không giao nào dài ≥ 12 → `[]`.

Câu hỏi thuật toán: làm sao tìm **giao sớm nhất đủ dài** mà không phải thử mọi cặp?

In [6]:
for x in S1:
    for y in S2:
        it = intersect(x, y)
        if it:
            print(x, "∩", y, "=", it, "dài", it[1] - it[0])

[10, 50] ∩ [0, 15] = [10, 15] dài 5
[60, 120] ∩ [60, 70] = [60, 70] dài 10


## 5. Cách 1 — Brute Force / baseline

### 5a. Thử mọi cặp khoảng

Với mọi `(x ∈ slots1, y ∈ slots2)`, tính phần giao; nếu dài `≥ duration` thì là ứng viên `[lo, lo + duration]`. Lấy ứng viên có `lo` nhỏ nhất.

In [7]:
def meeting_all_pairs(slots1, slots2, duration):
    best = None
    for a1, b1 in slots1:
        for a2, b2 in slots2:
            lo, hi = max(a1, a2), min(b1, b2)
            if hi - lo >= duration and (best is None or lo < best):
                best = lo
    return [] if best is None else [best, best + duration]


for s1, s2, d, want in examples:
    got = meeting_all_pairs(s1, s2, d)
    print(got)
    assert got == want

[60, 68]
[]


**Độ phức tạp:** Time `O(n·m)` — với `n = m = 10^4` là `10^8` cặp, quá chậm trong Python. Space `O(1)`.

### 5b. Trọng tài "quét từng thời điểm" (chỉ cho toạ độ nhỏ)

Một cách còn "ngây thơ" hơn nhưng **hiển nhiên đúng**: thử mọi `t = 0, 1, 2, ...`; `t` hợp lệ nếu `[t, t + duration]` nằm gọn trong một khung của người 1 **và** một khung của người 2. Dùng để kiểm tra các cách khác trên dữ liệu ngẫu nhiên nhỏ.

In [8]:
def meeting_scan_time(slots1, slots2, duration):
    def free(slots, t):
        return any(a <= t and t + duration <= b for a, b in slots)

    hi = max(b for _, b in slots1 + slots2)
    for t in range(0, hi + 1):
        if free(slots1, t) and free(slots2, t):
            return [t, t + duration]
    return []


for s1, s2, d, want in examples:
    assert meeting_scan_time(s1, s2, d) == want
print("trọng tài quét thời gian đúng trên ví dụ")

trọng tài quét thời gian đúng trên ví dụ


**Nút thắt của 5a:** phần lớn các cặp **không giao nhau** (khung 9 giờ sáng của người 1 chẳng liên quan gì khung 5 giờ chiều của người 2). Nếu sắp xếp theo thời gian, mỗi khung chỉ cần so với vài khung "gần" nó.

## 6. Bước đột phá

Sắp xếp cả hai danh sách theo `start`. Đặt con trỏ `i` (người 1) và `j` (người 2) ở đầu. Xét cặp hiện tại `slots1[i]`, `slots2[j]`:

- **Nếu** phần giao dài `≥ duration` **thì** trả về `[lo, lo + duration]` ngay — đó là sớm nhất (chứng minh ở mục 10).
- **Ngược lại**, bỏ khung **kết thúc sớm hơn**:
  - **Nếu** `slots1[i][1] < slots2[j][1]` **thì** `i += 1`, **ngược lại** `j += 1`.

**Tại sao bỏ khung kết thúc sớm hơn là an toàn?** Giả sử `slots1[i]` kết thúc sớm hơn (`e1 ≤ e2`). Mọi khung sau của người 2 (`slots2[j+1..]`) bắt đầu **sau** `e2 ≥ e1` (vì các khung của người 2 không giao nhau và đã sắp xếp), nên chúng **không giao** `slots1[i]`. Còn với `slots2[j]`, ta vừa kiểm tra xong — không đủ dài. Vậy `slots1[i]` hết giá trị, bỏ đi.

**Bất biến:** mọi cặp `(i', j')` với `i' < i` hoặc `j' < j` đều đã được xác định là không cho khung họp nào hợp lệ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + hai con trỏ (lời giải chuẩn)

In [9]:
def meeting_two_pointers(slots1, slots2, duration):
    slots1 = sorted(slots1)
    slots2 = sorted(slots2)
    i = j = 0
    while i < len(slots1) and j < len(slots2):
        lo = max(slots1[i][0], slots2[j][0])
        hi = min(slots1[i][1], slots2[j][1])
        if hi - lo >= duration:
            return [lo, lo + duration]
        if slots1[i][1] < slots2[j][1]:
            i += 1
        else:
            j += 1
    return []


for s1, s2, d, want in examples:
    assert meeting_two_pointers(s1, s2, d) == want
print("Cách 2 đúng trên ví dụ")

Cách 2 đúng trên ví dụ


**Độ phức tạp:** Time `O(n log n + m log m)` cho sắp xếp, cộng `O(n + m)` cho vòng lặp. Space `O(n + m)` (bản sao đã sắp xếp; có thể sắp xếp tại chỗ để `O(1)` phụ trội ngoài bộ nhớ của sort).

### 7.2 Cách 3 — Gộp hai danh sách, lọc khung ngắn, so sánh kề nhau

Ý tưởng: bỏ hết khung có độ dài `< duration` (không bao giờ dùng được). Gộp khung của **cả hai người** rồi sắp xếp theo `start`. Vì khung của cùng một người không giao nhau, **hai khung giao nhau chắc chắn thuộc hai người khác nhau**. Chỉ cần xét từng cặp **kề nhau** `(cur, nxt)`: nếu `cur.end >= nxt.start + duration` thì họp `[nxt.start, nxt.start + duration]`.

In [10]:
def meeting_merge(slots1, slots2, duration):
    allv = sorted(s for s in slots1 + slots2 if s[1] - s[0] >= duration)
    for k in range(len(allv) - 1):
        if allv[k][1] >= allv[k + 1][0] + duration:
            return [allv[k + 1][0], allv[k + 1][0] + duration]
    return []


for s1, s2, d, want in examples:
    assert meeting_merge(s1, s2, d) == want
print("Cách 3 đúng trên ví dụ")

Cách 3 đúng trên ví dụ


Tại sao `nxt.end` không cần kiểm tra? Vì đã lọc: `nxt.end − nxt.start >= duration` nên `nxt.end >= nxt.start + duration` sẵn rồi. Phần giao là `[nxt.start, min(cur.end, nxt.end)]` và cả hai đầu mút phải đều `>= nxt.start + duration`.

**Độ phức tạp:** Time `O((n + m) log(n + m))`, Space `O(n + m)`.

### 7.3 Cách 4 — Heap thay cho sắp xếp

Giống Cách 3 nhưng dùng min-heap để lấy lần lượt khung có `start` nhỏ nhất. Lợi ích: nếu đáp án xuất hiện sớm, ta **dừng sớm** sau vài lần `heappop` (`O(log N)` mỗi lần) thay vì phải sắp xếp toàn bộ `O(N log N)`; `heapify` chỉ `O(N)`.

In [11]:
def meeting_heap(slots1, slots2, duration):
    h = [(a, b) for a, b in slots1 + slots2 if b - a >= duration]
    heapq.heapify(h)
    while len(h) > 1:
        a1, b1 = heapq.heappop(h)
        a2, b2 = h[0]                    # khung kế tiếp theo start
        if b1 >= a2 + duration:
            return [a2, a2 + duration]
    return []


for s1, s2, d, want in examples:
    assert meeting_heap(s1, s2, d) == want
print("Cách 4 đúng trên ví dụ")

Cách 4 đúng trên ví dụ


**Độ phức tạp:** Time `O(N + k log N)` với `N = n + m` và `k` là số lần `heappop` trước khi tìm thấy (tệ nhất `k = N` → `O(N log N)`). Space `O(N)`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 (`duration = 8`). Sau khi sắp xếp: `slots1 = [[10,50],[60,120],[140,210]]`, `slots2 = [[0,15],[60,70]]`.

| Bước | i | slots1[i] | j | slots2[j] | lo | hi | hi − lo | Hành động |
|---|---|---|---|---|---|---|---|---|
| 1 | 0 | [10,50] | 0 | [0,15] | 10 | 15 | 5 | 15 < 50 → `j += 1` |
| 2 | 0 | [10,50] | 1 | [60,70] | 60 | 50 | −10 | 50 < 70 → `i += 1` |
| 3 | 1 | [60,120] | 1 | [60,70] | 60 | 70 | 10 | ≥ 8 → trả `[60,68]` |

Với `duration = 12`, bước 3 có `10 < 12` → 70 < 120 → `j += 1` → `j` hết → `[]`.

In [12]:
def trace_two_pointers(slots1, slots2, duration):
    slots1, slots2 = sorted(slots1), sorted(slots2)
    i = j = 0
    step = 1
    while i < len(slots1) and j < len(slots2):
        lo = max(slots1[i][0], slots2[j][0])
        hi = min(slots1[i][1], slots2[j][1])
        if hi - lo >= duration:
            print(f"bước {step}: i={i} {slots1[i]} j={j} {slots2[j]} lo={lo} hi={hi} dài={hi - lo} >= {duration} -> {[lo, lo + duration]}")
            return [lo, lo + duration]
        move = "i += 1" if slots1[i][1] < slots2[j][1] else "j += 1"
        print(f"bước {step}: i={i} {slots1[i]} j={j} {slots2[j]} lo={lo} hi={hi} dài={hi - lo} < {duration} -> {move}")
        if slots1[i][1] < slots2[j][1]:
            i += 1
        else:
            j += 1
        step += 1
    print("một con trỏ đã hết -> []")
    return []


for s1, s2, d, _ in examples:
    trace_two_pointers(s1, s2, d)
    print()

bước 1: i=0 [10, 50] j=0 [0, 15] lo=10 hi=15 dài=5 < 8 -> j += 1
bước 2: i=0 [10, 50] j=1 [60, 70] lo=60 hi=50 dài=-10 < 8 -> i += 1
bước 3: i=1 [60, 120] j=1 [60, 70] lo=60 hi=70 dài=10 >= 8 -> [60, 68]

bước 1: i=0 [10, 50] j=0 [0, 15] lo=10 hi=15 dài=5 < 12 -> j += 1
bước 2: i=0 [10, 50] j=1 [60, 70] lo=60 hi=50 dài=-10 < 12 -> i += 1
bước 3: i=1 [60, 120] j=1 [60, 70] lo=60 hi=70 dài=10 < 12 -> j += 1
một con trỏ đã hết -> []



Trace Cách 3 trên ví dụ 1 để thấy danh sách sau khi lọc và gộp:

In [13]:
def trace_merge(slots1, slots2, duration):
    allv = sorted(s for s in slots1 + slots2 if s[1] - s[0] >= duration)
    print("sau khi lọc (dài >= %d) và sắp xếp:" % duration, allv)
    for k in range(len(allv) - 1):
        cur, nxt = allv[k], allv[k + 1]
        ok = cur[1] >= nxt[0] + duration
        print(f"  so {cur} với {nxt}: {cur[1]} >= {nxt[0]} + {duration}? {ok}")
        if ok:
            return [nxt[0], nxt[0] + duration]
    return []


print(trace_merge(S1, S2, 8))

sau khi lọc (dài >= 8) và sắp xếp: [[0, 15], [10, 50], [60, 70], [60, 120], [140, 210]]
  so [0, 15] với [10, 50]: 15 >= 10 + 8? False
  so [10, 50] với [60, 70]: 50 >= 60 + 8? False
  so [60, 70] với [60, 120]: 70 >= 60 + 8? True
[60, 68]


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minAvailableDuration(self, slots1: List[List[int]], slots2: List[List[int]], duration: int) -> List[int]:
        slots1.sort()
        slots2.sort()
        i = j = 0
        m, n = len(slots1), len(slots2)
        while i < m and j < n:
            start = max(slots1[i][0], slots2[j][0])
            end = min(slots1[i][1], slots2[j][1])
            if end - start >= duration:
                return [start, start + duration]
            if slots1[i][1] < slots2[j][1]:
                i += 1
            else:
                j += 1
        return []


sol = Solution()
for s1, s2, d, want in examples:
    got = sol.minAvailableDuration([x[:] for x in s1], [x[:] for x in s2], d)
    print(got)
    assert got == want

[60, 68]
[]


## 10. Vì sao đúng

**Bổ đề (bỏ khung kết thúc sớm):** giả sử `e1 = slots1[i][1] ≤ e2 = slots2[j][1]` và cặp `(i, j)` không đủ dài. Với mọi `j' > j`, `slots2[j'][0] > e2 ≥ e1` (khung người 2 không giao nhau, đã sắp xếp) nên `slots1[i]` không giao `slots2[j']`. Với `j' < j`: theo bất biến, đã loại. Vậy `slots1[i]` không tham gia khung hợp lệ nào → bỏ an toàn. Trường hợp `e2 < e1` đối xứng (code dùng `else` cho cả trường hợp bằng nhau — khi `e1 == e2` bỏ bên nào cũng đúng).

**Sớm nhất:** giả sử cặp đầu tiên tìm thấy là `(i, j)` với khung bắt đầu `lo = max(s1_i, s2_j)`. Mọi cặp hợp lệ khác `(i', j')` có `i' ≥ i`, `j' ≥ j` (các cặp nhỏ hơn đã bị loại). Khi đó `max(s1_{i'}, s2_{j'}) ≥ max(s1_i, s2_j) = lo` vì danh sách tăng theo `start`. Vậy không có khung nào bắt đầu sớm hơn `lo`.

**Cách 3 (gộp):** xem chứng minh trong mục 6–7: hai khung chồng nhau đủ dài phải thuộc hai người khác nhau; nếu khung họp sớm nhất đến từ cặp `(a, b)` với `a.start ≤ b.start`, mọi khung `c` nằm giữa `a` và `b` trong thứ tự gộp sẽ hoặc giao `a` (vô lý nếu cùng người với `a`), hoặc là khung của người kia nằm gọn trong `a` và đủ dài (sau khi lọc) — tạo khung họp sớm hơn (vô lý). Nên `a`, `b` kề nhau.

## 11. Time / Space Complexity

Gọi `n = len(slots1)`, `m = len(slots2)`, `N = n + m`.

| Cách | Time | Space |
|---|---|---|
| 1a. Mọi cặp | `O(n·m)` | `O(1)` |
| 1b. Quét thời gian | `O(T · N)` với `T` = toạ độ lớn nhất | `O(1)` |
| 2. Sort + hai con trỏ | `O(n log n + m log m)` | `O(1)`–`O(N)` |
| 3. Gộp + kề nhau | `O(N log N)` | `O(N)` |
| 4. Heap | `O(N + k log N)` | `O(N)` |

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [15]:
methods = {
    "all_pairs": meeting_all_pairs,
    "two_pointers": meeting_two_pointers,
    "merge": meeting_merge,
    "heap": meeting_heap,
    "submit": lambda a, b, d: Solution().minAvailableDuration([x[:] for x in a], [x[:] for x in b], d),
}
edge = [
    ([[0, 10]], [[0, 10]], 10, [0, 10]),                    # vừa khít (dấu bằng)
    ([[0, 10]], [[0, 10]], 11, []),
    ([[0, 5]], [[5, 10]], 1, []),                           # chạm một điểm, dài 0
    ([[10, 20], [0, 5]], [[3, 15]], 2, [3, 5]),             # input chưa sắp xếp
    ([[0, 1000000000]], [[999999990, 1000000000]], 10, [999999990, 1000000000]),
    ([[0, 100]], [[10, 12], [20, 40]], 8, [20, 28]),        # khung ngắn nằm giữa
]
for name, f in methods.items():
    for s1, s2, d, want in examples + edge:
        assert f(s1, s2, d) == want, (name, s1, s2, d)
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên với trọng tài quét thời gian (seed cố định)

Sinh khung **không giao nhau** cho mỗi người: chọn ngẫu nhiên các điểm cắt phân biệt rồi ghép cặp liên tiếp, bỏ ngẫu nhiên vài khung, rồi xáo trộn thứ tự.

In [16]:
def random_slots(rng, max_t, k):
    pts = sorted(rng.sample(range(0, max_t + 1), 2 * k))
    slots = [[pts[2 * t], pts[2 * t + 1]] for t in range(k)]
    slots = [s for s in slots if rng.random() < 0.8] or slots[:1]
    rng.shuffle(slots)
    return slots


rng = random.Random(1229)
for _ in range(500):
    s1 = random_slots(rng, 60, rng.randint(1, 6))
    s2 = random_slots(rng, 60, rng.randint(1, 6))
    d = rng.randint(1, 15)
    want = meeting_scan_time(s1, s2, d)
    for name, f in methods.items():
        assert f(s1, s2, d) == want, (name, s1, s2, d)
print("500 test ngẫu nhiên: mọi cách khớp trọng tài")

500 test ngẫu nhiên: mọi cách khớp trọng tài


### 12.3 Test giới hạn (`10^4` khung mỗi người)

In [17]:
n = 10 ** 4
big1 = [[20 * k, 20 * k + 9] for k in range(n)]               # rảnh [20k, 20k+9]
big2 = [[20 * k + 10, 20 * k + 19] for k in range(n)]         # rảnh xen kẽ, không bao giờ trùng
big2[-1] = [20 * (n - 1) + 2, 20 * (n - 1) + 8]                # chỉ khung cuối giao, dài 6
rng.shuffle(big1)
rng.shuffle(big2)
for name in ["two_pointers", "merge", "heap", "submit"]:
    t0 = time.perf_counter()
    r = methods[name](big1, big2, 6)
    r_none = methods[name](big1, big2, 7)
    dt = time.perf_counter() - t0
    assert r == [20 * (n - 1) + 2, 20 * (n - 1) + 8] and r_none == []
    print(f"{name:13s} {r} trong {dt * 1000:.1f} ms")

two_pointers  [199982, 199988] trong 26.3 ms
merge         [199982, 199988] trong 24.8 ms


heap          [199982, 199988] trong 36.5 ms


submit        [199982, 199988] trong 31.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp (đề không hứa input đã sắp xếp)

In [18]:
def wrong_no_sort(slots1, slots2, duration):
    i = j = 0
    while i < len(slots1) and j < len(slots2):
        lo, hi = max(slots1[i][0], slots2[j][0]), min(slots1[i][1], slots2[j][1])
        if hi - lo >= duration:
            return [lo, lo + duration]
        if slots1[i][1] < slots2[j][1]:
            i += 1
        else:
            j += 1
    return []


s1, s2 = [[60, 120], [10, 50]], [[0, 15], [12, 30]]
print("sai :", wrong_no_sort(s1, s2, 5))
print("đúng:", meeting_two_pointers(s1, s2, 5))

sai : []
đúng: [10, 15]


**Vì sao sai:** lập luận "khung sau của người kia bắt đầu sau khung hiện tại" chỉ đúng khi đã sắp xếp. Ở đây `[60,120]` đứng đầu `slots1`; nó kết thúc muộn hơn cả `[0,15]` lẫn `[12,30]` nên con trỏ `j` bỏ qua cả hai — `[10,50]` không bao giờ được so với `[12,30]` (giao `[12,30]` dài 18). **Sửa:** `sort()` cả hai trước.

### Lỗi 2 — Di chuyển nhầm con trỏ (bỏ khung kết thúc **muộn** hơn)

In [19]:
def wrong_move(slots1, slots2, duration):
    slots1, slots2 = sorted(slots1), sorted(slots2)
    i = j = 0
    while i < len(slots1) and j < len(slots2):
        lo, hi = max(slots1[i][0], slots2[j][0]), min(slots1[i][1], slots2[j][1])
        if hi - lo >= duration:
            return [lo, lo + duration]
        if slots1[i][1] > slots2[j][1]:      # đảo dấu!
            i += 1
        else:
            j += 1
    return []


print("sai :", wrong_move(S1, S2, 8))
print("đúng:", meeting_two_pointers(S1, S2, 8))

sai : []
đúng: [60, 68]


**Vì sao sai:** bỏ `[10,50]` (kết thúc muộn hơn `[0,15]`)... rồi lại bỏ `[60,120]`, khung có thể giao với `[60,70]` bị mất. Khung kết thúc muộn hơn vẫn còn "tương lai" để giao với khung tiếp theo của người kia. **Sửa:** bỏ khung có `end` **nhỏ hơn**.

### Lỗi 3 — Dùng `>` thay vì `>=` (bỏ trường hợp vừa khít)

In [20]:
def wrong_strict(slots1, slots2, duration):
    slots1, slots2 = sorted(slots1), sorted(slots2)
    i = j = 0
    while i < len(slots1) and j < len(slots2):
        lo, hi = max(slots1[i][0], slots2[j][0]), min(slots1[i][1], slots2[j][1])
        if hi - lo > duration:
            return [lo, lo + duration]
        if slots1[i][1] < slots2[j][1]:
            i += 1
        else:
            j += 1
    return []


print("sai :", wrong_strict(S1, S2, 10))     # giao [60,70] dài đúng 10
print("đúng:", meeting_two_pointers(S1, S2, 10))

sai : []
đúng: [60, 70]


**Vì sao sai:** khung họp `[60, 70]` dài đúng 10 nằm gọn trong giao `[60, 70]` — hợp lệ. **Sửa:** `hi - lo >= duration`.

### Lỗi 4 — Cách gộp nhưng quên lọc khung ngắn

In [21]:
def wrong_merge_no_filter(slots1, slots2, duration):
    allv = sorted(slots1 + slots2)
    for k in range(len(allv) - 1):
        if allv[k][1] >= allv[k + 1][0] + duration:
            return [allv[k + 1][0], allv[k + 1][0] + duration]
    return []


s1, s2 = [[0, 100]], [[10, 12], [20, 40]]
print("sai :", wrong_merge_no_filter(s1, s2, 8))    # [10, 18] — nhưng người 2 chỉ rảnh [10, 12]!
print("đúng:", meeting_merge(s1, s2, 8))

sai : [10, 18]
đúng: [20, 28]


**Vì sao sai:** điều kiện chỉ kiểm tra `cur.end`, ngầm giả định `nxt` đủ dài. Khung `[10,12]` của người 2 quá ngắn nên `[10,18]` vượt ra ngoài. **Sửa:** lọc `end − start >= duration` trước khi gộp (hoặc kiểm tra cả `nxt.end`).

### Lỗi 5 — Trả về cả phần giao `[lo, hi]`

Đề yêu cầu khung **dài đúng `duration`**: `[lo, lo + duration]`, không phải `[60, 70]`.

## 14. Mẹo & nhận dạng pattern

- Hai danh sách khoảng **không tự giao** → **sắp xếp + hai con trỏ**, bỏ khoảng **kết thúc sớm hơn** (giống Interval List Intersections — LeetCode 986).
- "Sớm nhất / đầu tiên thoả" + duyệt theo thứ tự thời gian → **trả về ngay** khi gặp.
- Gộp nhiều nguồn theo thời gian → **heap** hoặc sort chung; nếu mỗi nguồn không tự giao thì chỉ cần so **phần tử kề nhau**.
- Lọc bỏ sớm các phần tử không bao giờ dùng được (khung ngắn hơn `duration`) giúp đơn giản hoá điều kiện.
- Cẩn thận dấu `>=` với khoảng **đóng**.

## 15. Biến thể

1. **Interval List Intersections (LeetCode 986):** trả về **mọi** phần giao — cùng khung hai con trỏ, không dừng sớm.
2. **`k` người:** gộp lần lượt (giao của danh sách A và B là một danh sách không tự giao, giao tiếp với C...), hoặc quét sự kiện đếm số người rảnh.
3. **Trả về khung muộn nhất:** duyệt ngược từ cuối.
4. **Nhiều phòng họp / Meeting Rooms II (LeetCode 253):** quét sự kiện + heap.
5. Khung **nửa mở** `[start, end)`: điều kiện vẫn là `end − start >= duration`, nhưng "chạm một điểm" không còn là giao.

In [22]:
def all_intersections(slots1, slots2):
    """Biến thể 1: mọi phần giao (độ dài >= 0) của hai danh sách."""
    a, b = sorted(slots1), sorted(slots2)
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        lo, hi = max(a[i][0], b[j][0]), min(a[i][1], b[j][1])
        if lo <= hi:
            out.append([lo, hi])
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return out


print(all_intersections(S1, S2))
print(all_intersections([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]]))

[[10, 15], [60, 70]]
[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Mọi cặp | thử hết `(x, y)` | `O(n·m)` | `O(1)` | trọng tài |
| 1b. Quét thời gian | thử mọi `t` | `O(T·N)` | `O(1)` | trọng tài toạ độ nhỏ |
| 2. Sort + hai con trỏ | bỏ khung kết thúc sớm | `O(n log n + m log m)` | `O(1)` | **nộp bài** |
| 3. Gộp + kề nhau | lọc ngắn, so kề | `O(N log N)` | `O(N)` | code ngắn, mở rộng nhiều nguồn |
| 4. Heap | như 3, dừng sớm | `O(N + k log N)` | `O(N)` | đáp án thường xuất hiện sớm |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`Solution().minAvailableDuration([[0, 2], [3, 10]], [[1, 4], [6, 20]], 3)` = ?

### Bài 2 (tính tay)

Trace hai con trỏ cho `slots1 = [[5, 9], [20, 30]]`, `slots2 = [[0, 6], [8, 25]]`, `duration = 4`.

### Bài 3 (điền TODO) — khung muộn nhất

In [23]:
def latest_meeting(slots1, slots2, duration):
    """Trả về khung [end - duration, end] MUỘN nhất mà cả hai rảnh, hoặc []."""
    a, b = sorted(slots1), sorted(slots2)
    i, j = len(a) - 1, len(b) - 1
    # TODO: duyệt ngược; bỏ khung BẮT ĐẦU muộn hơn
    raise NotImplementedError

# print(latest_meeting([[10,50],[60,120],[140,210]], [[0,15],[60,70]], 8))   # mong đợi [62, 70]

### Bài 4 (tự viết lại)

Viết lại Cách 2 nhưng **không sửa** input và **không tạo bản sao**: sắp xếp danh sách **chỉ số** theo `start`.

In [24]:
def meeting_index_sort_todo(slots1, slots2, duration):
    # TODO: order1 = sorted(range(len(slots1)), key=lambda k: slots1[k][0]) ...
    raise NotImplementedError

# print(meeting_index_sort_todo([[10,50],[60,120],[140,210]], [[0,15],[60,70]], 8))   # [60, 68]

### Bài 5 (mini challenge) — `k` người

Cho danh sách `people` gồm `k` danh sách khung (mỗi người không tự giao), tìm khung sớm nhất dài `duration` cho **tất cả**.

In [25]:
def meeting_k_people_todo(people, duration):
    # TODO: gợi ý — giao lần lượt bằng all_intersections, rồi tìm phần giao đầu tiên đủ dài
    raise NotImplementedError

# print(meeting_k_people_todo([[[0, 30]], [[5, 20]], [[10, 40]]], 5))   # mong đợi [10, 15]

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: giao các cặp theo thứ tự thời gian. Mức 2: `[0,2]∩[1,4] = [1,2]` (dài 1), `[3,10]∩[1,4] = [3,4]` (dài 1), `[3,10]∩[6,20] = [6,10]` (dài 4). Mức 3: `[6, 9]`.

**Bài 2.** Mức 1: `[5,9]∩[0,6] = [5,6]`, dài 1, `6 < 9` → `j += 1`. Mức 2: `[5,9]∩[8,25] = [8,9]`, dài 1, `9 < 25` → `i += 1`. Mức 3: `[20,30]∩[8,25] = [20,25]`, dài 5 ≥ 4 → `[20, 24]`.

**Bài 3.** Mức 1: phần giao `[lo, hi]`, khung muộn nhất là `[hi − duration, hi]`. Mức 2: nếu `a[i][0] > b[j][0]` thì `i -= 1`, ngược lại `j -= 1`. Mức 3: dừng khi một chỉ số `< 0`.

**Bài 4.** Mức 1: hai list chỉ số đã sắp xếp. Mức 2: con trỏ `p`, `q` chạy trên list chỉ số, đọc `slots1[order1[p]]`. Mức 3: phần còn lại y hệt Cách 2.

**Bài 5.** Mức 1: `common = people[0]`. Mức 2: `common = all_intersections(common, people[t])` cho `t = 1..k−1` (kết quả vẫn không tự giao). Mức 3: duyệt `common` đã sắp xếp, trả khung đầu tiên có `hi − lo >= duration`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1, 2
r1 = Solution().minAvailableDuration([[0, 2], [3, 10]], [[1, 4], [6, 20]], 3)
print(r1)
assert r1 == [6, 9]
r2 = trace_two_pointers([[5, 9], [20, 30]], [[0, 6], [8, 25]], 4)
assert r2 == [20, 24]

[6, 9]
bước 1: i=0 [5, 9] j=0 [0, 6] lo=5 hi=6 dài=1 < 4 -> j += 1
bước 2: i=0 [5, 9] j=1 [8, 25] lo=8 hi=9 dài=1 < 4 -> i += 1
bước 3: i=1 [20, 30] j=1 [8, 25] lo=20 hi=25 dài=5 >= 4 -> [20, 24]


In [27]:
def latest_meeting(slots1, slots2, duration):
    a, b = sorted(slots1), sorted(slots2)
    i, j = len(a) - 1, len(b) - 1
    while i >= 0 and j >= 0:
        lo, hi = max(a[i][0], b[j][0]), min(a[i][1], b[j][1])
        if hi - lo >= duration:
            return [hi - duration, hi]
        if a[i][0] > b[j][0]:
            i -= 1
        else:
            j -= 1
    return []


def meeting_index_sort(slots1, slots2, duration):
    o1 = sorted(range(len(slots1)), key=lambda k: slots1[k][0])
    o2 = sorted(range(len(slots2)), key=lambda k: slots2[k][0])
    p = q = 0
    while p < len(o1) and q < len(o2):
        x, y = slots1[o1[p]], slots2[o2[q]]
        lo, hi = max(x[0], y[0]), min(x[1], y[1])
        if hi - lo >= duration:
            return [lo, lo + duration]
        if x[1] < y[1]:
            p += 1
        else:
            q += 1
    return []


def meeting_k_people(people, duration):
    common = sorted(people[0])
    for t in range(1, len(people)):
        common = all_intersections(common, people[t])
    for lo, hi in common:
        if hi - lo >= duration:
            return [lo, lo + duration]
    return []


def latest_scan(slots1, slots2, duration):
    hi = max(b for _, b in slots1 + slots2)
    for t in range(hi - duration, -1, -1):
        if any(a <= t and t + duration <= b for a, b in slots1) and any(a <= t and t + duration <= b for a, b in slots2):
            return [t, t + duration]
    return []


assert latest_meeting(S1, S2, 8) == [62, 70]
assert meeting_index_sort(S1, S2, 8) == [60, 68]
assert meeting_k_people([[[0, 30]], [[5, 20]], [[10, 40]]], 5) == [10, 15]
rng = random.Random(77)
for _ in range(300):
    s1 = random_slots(rng, 50, rng.randint(1, 5))
    s2 = random_slots(rng, 50, rng.randint(1, 5))
    s3 = random_slots(rng, 50, rng.randint(1, 5))
    d = rng.randint(1, 10)
    assert latest_meeting(s1, s2, d) == latest_scan(s1, s2, d)
    before = [x[:] for x in s1]
    assert meeting_index_sort(s1, s2, d) == meeting_scan_time(s1, s2, d) and s1 == before
    want3 = []
    for t in range(0, 51):
        if all(any(a <= t and t + d <= b for a, b in s) for s in (s1, s2, s3)):
            want3 = [t, t + d]
            break
    assert meeting_k_people([s1, s2, s3], d) == want3
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
slots1.sort(); slots2.sort()
i = j = 0
while i < len(slots1) and j < len(slots2):
    lo = max(slots1[i][0], slots2[j][0])
    hi = min(slots1[i][1], slots2[j][1])
    if hi - lo >= duration:            # >= (khoảng đóng, vừa khít vẫn được)
        return [lo, lo + duration]
    if slots1[i][1] < slots2[j][1]:    # bỏ khung KẾT THÚC SỚM hơn
        i += 1
    else:
        j += 1
return []
```

**Checklist:**
- [ ] Đã `sort` cả hai danh sách?
- [ ] Bỏ khung có `end` nhỏ hơn?
- [ ] So sánh `>= duration`?
- [ ] Trả `[lo, lo + duration]` (không phải `[lo, hi]`)?
- [ ] Nếu dùng cách gộp: đã lọc khung ngắn hơn `duration`?
- [ ] Thử: chạm một điểm, vừa khít, input chưa sắp xếp, không có đáp án?